In [ ]:
import sys
from IPython.display import display, JSON

print(sys.executable)
print(sys.version)

In [ ]:
from IPython import get_ipython
from IPython.lib.pretty import pretty

ip = get_ipython()

ip.display_formatter.formatters["text/plain"].for_type(
    dict, lambda obj, p, cycle: p.text(pretty(obj))
)

ip.display_formatter.formatters["text/plain"].for_type(
    list, lambda obj, p, cycle: p.text(pretty(obj))
)

In [ ]:
from dotenv import load_dotenv

load_dotenv("../.env")

In [ ]:
import sys
import importlib.metadata

print("Python:", sys.version)
print("Executable:", sys.executable)

print(
    "LangChain Groq:",
    importlib.metadata.version("langchain-groq")
)

from langchain_groq import ChatGroq

print("Import successful!")

In [ ]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv("../.env")

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_retries=3,
    timeout=60
)

response = llm.invoke(
    "Review this Python function for potential bugs: "
    "def divide(a, b): return a / b"
)

print(response.content)

In [ ]:
%pip install requests python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
GITHUB_OWNER = os.getenv("GITHUB_OWNER")
GITHUB_REPO = os.getenv("GITHUB_REPO")
GITHUB_PR_NUMBER = int(os.getenv("GITHUB_PR_NUMBER"))

print("Owner:", GITHUB_OWNER)
print("Repository:", GITHUB_REPO)
print("PR number:", GITHUB_PR_NUMBER)
print("Token loaded:", bool(GITHUB_TOKEN))

In [ ]:
import requests

url = (
    f"https://api.github.com/repos/"
    f"{GITHUB_OWNER}/{GITHUB_REPO}"
)

headers = {
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json"
}

response = requests.get(
    url,
    headers=headers,
    timeout=30
)

response.raise_for_status()

repo = response.json()

print("Repository:", repo["full_name"])
print("Private:", repo["private"])
print("Default branch:", repo["default_branch"])

In [ ]:
import requests

def list_pull_requests(owner, repo, token, state="open", head=None, base=None):
    """ Fetch pull requests from a GitHub repository.

    Args:
        owner: GitHub username
        repo: Repository name
        token: GitHub personal access token
        state: open, closed, or all
        head: Source branch (optional)
        base: Target branch (optional)

    Returns:
        List of PRs with relevant metadata.
    """

    url = f"https://api.github.com/repos/{owner}/{repo}/pulls"

    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github+json"
    }

    params = {
        "state": state,
        "per_page": 100
    }

    if head:
        params["head"] = f"{owner}:{head}"

    if base:
        params["base"] = base

    pull_requests = []

    while url:
        response = requests.get(
            url,
            headers=headers,
            params=params,
            timeout=30
        )

        response.raise_for_status()

        for pr in response.json():
            pull_requests.append(pr)
            pull_requests.append({
                "number": pr["number"],
                "title": pr["title"],
                "state": pr["state"],
                "source_branch": pr["head"]["ref"],
                "target_branch": pr["base"]["ref"],
                "author": pr["user"]["login"],
                "created_at": pr["created_at"],
                "url": pr["html_url"]
            })

        # GitHub pagination
        url = response.links.get("next", {}).get("url")
        params = None

    return pull_requests

prs = list_pull_requests(
    owner=os.getenv("GITHUB_OWNER"),
    repo=os.getenv("GITHUB_REPO"),
    token=os.getenv("GITHUB_TOKEN"),
    state="open",
    head="feature/user-profile-route",
    base="main"
)

for pr in prs:
    print(pr)

In [ ]:
prs

In [ ]:
def get_pull_request(owner, repo, token, pr_number):
    """
    Fetch details of a specific GitHub pull request.
    """

    url = (
        f"https://api.github.com/repos/"
        f"{owner}/{repo}/pulls/{pr_number}"
    )

    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github+json"
    }

    response = requests.get(
        url,
        headers=headers,
        timeout=30
    )

    response.raise_for_status()
    pr = response.json()

    return {
        "number": pr["number"],
        "title": pr["title"],
        "description": pr["body"],
        "state": pr["state"],
        "author": pr["user"]["login"],
        "url": pr["html_url"],

        "source_branch": pr["head"]["ref"],
        "target_branch": pr["base"]["ref"],

        "head_sha": pr["head"]["sha"],
        "base_sha": pr["base"]["sha"],

        "head_repo": (
            pr["head"]["repo"]["full_name"]
            if pr["head"]["repo"] else None
        ),

        "changed_files": pr["changed_files"],
        "additions": pr["additions"],
        "deletions": pr["deletions"],
        "commits": pr["commits"],

        "created_at": pr["created_at"],
        "updated_at": pr["updated_at"]
    }

pr = get_pull_request(
    owner=os.getenv("GITHUB_OWNER"),
    repo=os.getenv("GITHUB_REPO"),
    token=os.getenv("GITHUB_TOKEN"),
    pr_number=int(os.getenv("GITHUB_PR_NUMBER"))
)

display(JSON(pr, expanded=False))

In [ ]:
def get_pr_diff(owner, repo, token, pr_number):
    """
    Fetch the complete unified diff of a GitHub PR.
    """

    url = (
        f"https://api.github.com/repos/"
        f"{owner}/{repo}/pulls/{pr_number}"
    )

    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github.diff"
    }

    response = requests.get(
        url,
        headers=headers,
        timeout=30
    )

    response.raise_for_status()

    return response.text

diff = get_pr_diff(
    owner=os.getenv("GITHUB_OWNER"),
    repo=os.getenv("GITHUB_REPO"),
    token=os.getenv("GITHUB_TOKEN"),
    pr_number=int(os.getenv("GITHUB_PR_NUMBER"))
)

print(diff)

In [ ]:
def get_pr_changed_files(owner, repo, token, pr_number):
    """
    Fetch all changed files and their patches for a GitHub PR.
    """

    url = (
        f"https://api.github.com/repos/"
        f"{owner}/{repo}/pulls/{pr_number}/files"
    )

    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github+json"
    }

    params = {"per_page": 100}
    changed_files = []

    while url:
        response = requests.get(
            url,
            headers=headers,
            params=params,
            timeout=30
        )

        response.raise_for_status()

        for file in response.json():
            changed_files.append({
                "filename": file["filename"],
                "status": file["status"],
                "sha": file["sha"],
                "additions": file["additions"],
                "deletions": file["deletions"],
                "changes": file["changes"],
                "patch": file.get("patch"),
                "raw_url": file.get("raw_url"),
                "previous_filename": file.get("previous_filename")
            })

        # Fetch the next page, if available
        url = response.links.get("next", {}).get("url")
        params = None

    return changed_files

changed_files = get_pr_changed_files(
    owner=os.getenv("GITHUB_OWNER"),
    repo=os.getenv("GITHUB_REPO"),
    token=os.getenv("GITHUB_TOKEN"),
    pr_number=int(os.getenv("GITHUB_PR_NUMBER"))
)

# JSON(changed_files, expanded=False)
changed_files